In [1]:
import os
import sys

import requests
from dotenv import load_dotenv


load_dotenv()

FINNHUB_API_KEY = os.getenv("FINNHUB_API_KEY")
QUOTE_URL = "https://finnhub.io/api/v1/quote"


def main() -> None:
    if not FINNHUB_API_KEY:
        raise ValueError(
            "FINNHUB_API_KEY is missing. Add it to the .env file."
        )

    try:
        response = requests.get(
            QUOTE_URL,
            params={
                "symbol": "AAPL",
                "token": FINNHUB_API_KEY,
            },
            timeout=15,
        )

        response.raise_for_status()
        quote = response.json()

    except requests.RequestException as exc:
        print(f"Finnhub connection failed: {exc}")
        sys.exit(1)

    print("Finnhub connection successful.")
    print("AAPL quote response:")
    print(quote)

    required_fields = {"c", "d", "dp", "h", "l", "o", "pc", "t"}

    if required_fields.issubset(quote):
        print("Quote response contains the expected fields.")
    else:
        print("Warning: quote response is missing expected fields.")


if __name__ == "__main__":
    main()

Finnhub connection successful.
AAPL quote response:
{'c': 308.91, 'd': -24.52, 'dp': -7.3539, 'h': 310.69, 'l': 300, 'o': 304.81, 'pc': 333.43, 't': 1785528000}
Quote response contains the expected fields.


In [2]:
import json
import os

import websocket
from dotenv import load_dotenv


load_dotenv()

FINNHUB_API_KEY = os.getenv("FINNHUB_API_KEY")

SYMBOLS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "TSLA",
    "AMZN",
]


def on_open(ws: websocket.WebSocketApp) -> None:
    """Subscribe to the selected stock symbols after connecting."""

    print("Connected to Finnhub WebSocket.")

    for symbol in SYMBOLS:
        subscription_message = {
            "type": "subscribe",
            "symbol": symbol,
        }

        ws.send(json.dumps(subscription_message))
        print(f"Subscription sent: {symbol}")


def on_message(
    ws: websocket.WebSocketApp,
    message: str,
) -> None:
    """Print each raw WebSocket message received from Finnhub."""

    try:
        payload = json.loads(message)

        print("\nRaw Finnhub message:")
        print(json.dumps(payload, indent=2))

    except json.JSONDecodeError:
        print("\nReceived non-JSON message:")
        print(message)


def on_error(
    ws: websocket.WebSocketApp,
    error: Exception,
) -> None:
    print(f"WebSocket error: {error}")


def on_close(
    ws: websocket.WebSocketApp,
    close_status_code,
    close_message,
) -> None:
    print(
        "Finnhub WebSocket closed "
        f"| status={close_status_code} "
        f"| message={close_message}"
    )


def main() -> None:
    if not FINNHUB_API_KEY:
        raise ValueError(
            "FINNHUB_API_KEY is missing from the .env file."
        )

    websocket_url = (
        f"wss://ws.finnhub.io?token={FINNHUB_API_KEY}"
    )

    websocket_app = websocket.WebSocketApp(
        websocket_url,
        on_open=on_open,
        on_message=on_message,
        on_error=on_error,
        on_close=on_close,
    )

    print("Connecting to Finnhub...")

    websocket_app.run_forever(
        ping_interval=20,
        ping_timeout=10,
    )


if __name__ == "__main__":
    try:
        main()

    except KeyboardInterrupt:
        print("\nWebSocket stopped by user.")

Connecting to Finnhub...
Connected to Finnhub WebSocket.
Subscription sent: AAPL
Subscription sent: MSFT
Subscription sent: NVDA
Subscription sent: TSLA
Subscription sent: AMZN

Raw Finnhub message:
{
  "type": "ping"
}
WebSocket error: 
Finnhub WebSocket closed | status=None | message=None


In [3]:
import json
import logging
import os
import socket
import uuid
from datetime import datetime, timezone

import websocket
from confluent_kafka import KafkaException, Producer
from dotenv import load_dotenv


load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger(__name__)


FINNHUB_API_KEY = os.getenv("FINNHUB_API_KEY")
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS")
KAFKA_RAW_TRADES_TOPIC = os.getenv(
    "KAFKA_RAW_TRADES_TOPIC",
    "stock-trades-raw",
)

SYMBOLS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "TSLA",
    "AMZN",
]


def validate_configuration() -> None:
    required_variables = {
        "FINNHUB_API_KEY": FINNHUB_API_KEY,
        "KAFKA_BOOTSTRAP_SERVERS": KAFKA_BOOTSTRAP_SERVERS,
        "KAFKA_RAW_TRADES_TOPIC": KAFKA_RAW_TRADES_TOPIC,
    }

    missing_variables = [
        name
        for name, value in required_variables.items()
        if not value
    ]

    if missing_variables:
        raise ValueError(
            "Missing environment variables: "
            + ", ".join(missing_variables)
        )


def build_kafka_producer() -> Producer:
    producer_config = {
        "bootstrap.servers": KAFKA_BOOTSTRAP_SERVERS,
        "client.id": f"finnhub-trade-producer-{socket.gethostname()}",
        "acks": "all",
        "enable.idempotence": True,
        "linger.ms": 50,
        "delivery.timeout.ms": 120000,
        "request.timeout.ms": 30000,
    }

    return Producer(producer_config)


kafka_producer = build_kafka_producer()


def delivery_report(error, message) -> None:
    if error is not None:
        logger.error(
            "Kafka delivery failed | error=%s",
            error,
        )
        return

    message_key = (
        message.key().decode("utf-8")
        if message.key()
        else None
    )

    logger.info(
        "Delivered | symbol=%s | partition=%s | offset=%s",
        message_key,
        message.partition(),
        message.offset(),
    )


def normalize_trade(trade: dict) -> dict:
    """
    Convert Finnhub's abbreviated trade structure into a clearer,
    portfolio-friendly event schema.
    """

    event_timestamp_ms = trade.get("t")

    if event_timestamp_ms is not None:
        event_timestamp_utc = datetime.fromtimestamp(
            event_timestamp_ms / 1000,
            tz=timezone.utc,
        ).isoformat()
    else:
        event_timestamp_utc = None

    return {
        "schema_version": "1.0",
        "event_id": str(uuid.uuid4()),
        "event_type": "trade",
        "source": "finnhub",
        "symbol": trade.get("s"),
        "price": trade.get("p"),
        "volume": trade.get("v"),
        "trade_conditions": trade.get("c", []),
        "event_timestamp_ms": event_timestamp_ms,
        "event_timestamp_utc": event_timestamp_utc,
        "ingestion_timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat(),
    }


def publish_trade(trade_event: dict) -> None:
    symbol = trade_event.get("symbol")

    if not symbol:
        logger.warning(
            "Skipping trade because symbol is missing | event=%s",
            trade_event,
        )
        return

    message_value = json.dumps(trade_event).encode("utf-8")
    message_key = symbol.encode("utf-8")

    while True:
        try:
            kafka_producer.produce(
                topic=KAFKA_RAW_TRADES_TOPIC,
                key=message_key,
                value=message_value,
                callback=delivery_report,
            )

            kafka_producer.poll(0)
            break

        except BufferError:
            logger.warning(
                "Kafka producer queue is full. Waiting for delivery."
            )
            kafka_producer.poll(1)

        except KafkaException:
            logger.exception(
                "Kafka error while publishing trade | symbol=%s",
                symbol,
            )
            raise


def on_open(ws: websocket.WebSocketApp) -> None:
    logger.info("Connected to Finnhub WebSocket.")

    for symbol in SYMBOLS:
        subscription_message = {
            "type": "subscribe",
            "symbol": symbol,
        }

        ws.send(json.dumps(subscription_message))
        logger.info("Subscribed to %s", symbol)


def on_message(
    ws: websocket.WebSocketApp,
    message: str,
) -> None:
    try:
        payload = json.loads(message)

    except json.JSONDecodeError:
        logger.warning(
            "Ignoring non-JSON Finnhub message | message=%s",
            message,
        )
        return

    message_type = payload.get("type")

    if message_type == "ping":
        logger.debug("Finnhub ping received.")
        return

    if message_type != "trade":
        logger.info(
            "Ignoring Finnhub message type | type=%s",
            message_type,
        )
        return

    trades = payload.get("data", [])

    logger.info(
        "Received Finnhub trade batch | records=%s",
        len(trades),
    )

    for raw_trade in trades:
        normalized_trade = normalize_trade(raw_trade)
        publish_trade(normalized_trade)


def on_error(
    ws: websocket.WebSocketApp,
    error,
) -> None:
    logger.error("Finnhub WebSocket error | error=%s", error)


def on_close(
    ws: websocket.WebSocketApp,
    close_status_code,
    close_message,
) -> None:
    logger.warning(
        "Finnhub WebSocket closed | status=%s | message=%s",
        close_status_code,
        close_message,
    )

    remaining_messages = kafka_producer.flush(timeout=15)

    if remaining_messages:
        logger.warning(
            "%s Kafka message(s) remained undelivered.",
            remaining_messages,
        )

""" def test_kafka_publish() -> None:
    # Publish one Finnhub-shaped sample trade for pipeline testing.

    sample_finnhub_trade = {
        "c": ["TEST"],
        "p": 214.75,
        "s": "AAPL",
        "t": int(datetime.now(timezone.utc).timestamp() * 1000),
        "v": 100,
    }

    normalized_trade = normalize_trade(sample_finnhub_trade)

    logger.info(
        "Publishing sample trade | event=%s",
        normalized_trade,
    )

    publish_trade(normalized_trade)

    remaining_messages = kafka_producer.flush(timeout=15)

    if remaining_messages == 0:
        logger.info("Sample trade delivered successfully.")
    else:
        logger.error(
            "%s sample message(s) were not delivered.",
            remaining_messages,
        ) """""

def main() -> None:
    validate_configuration()

    websocket_url = (
        f"wss://ws.finnhub.io?token={FINNHUB_API_KEY}"
    )

    websocket_app = websocket.WebSocketApp(
        websocket_url,
        on_open=on_open,
        on_message=on_message,
        on_error=on_error,
        on_close=on_close,
    )

    logger.info(
        "Starting Finnhub producer | Kafka=%s | topic=%s",
        KAFKA_BOOTSTRAP_SERVERS,
        KAFKA_RAW_TRADES_TOPIC,
    )

    websocket_app.run_forever(
        ping_interval=20,
        ping_timeout=10,
    )


if __name__ == "__main__":
    try:
        main()

    except KeyboardInterrupt:
        logger.info("Producer stopped by user.")

        remaining_messages = kafka_producer.flush(timeout=15)

        logger.info(
            "Kafka producer closed | undelivered=%s",
            remaining_messages,
        )

    except Exception:
        logger.exception(
            "Finnhub trade producer terminated unexpectedly."
        )
        raise

2026-08-05 15:22:37,496 | INFO | Starting Finnhub producer | Kafka=54.173.125.249:9093 | topic=stock-trades-raw
2026-08-05 15:22:37,669 | INFO | Websocket connected
2026-08-05 15:22:37,671 | INFO | Connected to Finnhub WebSocket.
2026-08-05 15:22:37,673 | INFO | Subscribed to AAPL
2026-08-05 15:22:37,675 | INFO | Subscribed to MSFT
2026-08-05 15:22:37,677 | INFO | Subscribed to NVDA
2026-08-05 15:22:37,678 | INFO | Subscribed to TSLA
2026-08-05 15:22:37,680 | INFO | Subscribed to AMZN
2026-08-05 15:22:37,801 | INFO | Received Finnhub trade batch | records=2
2026-08-05 15:22:37,914 | INFO | Received Finnhub trade batch | records=1
2026-08-05 15:22:38,296 | INFO | Received Finnhub trade batch | records=1
2026-08-05 15:22:38,298 | INFO | Delivered | symbol=MSFT | partition=0 | offset=4033
2026-08-05 15:22:38,299 | INFO | Delivered | symbol=MSFT | partition=0 | offset=4034
2026-08-05 15:22:38,301 | INFO | Delivered | symbol=TSLA | partition=0 | offset=4035
2026-08-05 15:22:39,412 | INFO | 